In [58]:
train=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
test=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")
ss=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
import xgboost
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

decision tree

In [3]:
train.columns

Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need'],
      dtype='object')

In [4]:
def clean_dataset():
    x_train=pd.get_dummies(train.iloc[0:10000][test.columns]).drop(columns=['id'])
    y_train=train.iloc[0:10000]['Irrigation_Need']
    x_test=pd.get_dummies(train.iloc[10000:20000][test.columns]).drop(columns=['id'])
    y_test=train.iloc[10000:20000]['Irrigation_Need']
    return x_train,y_train,x_test,y_test

In [5]:
def createDecisionTree(x_train,y_train):
    clf_gini = DecisionTreeClassifier(criterion="gini", random_state=100, max_depth=10, min_samples_leaf=5)
    clf_gini.fit(x_train,y_train)
    return clf_gini

In [8]:
def return_prediction(x_test,model):
    return model.predict(x_test)

def print_accuracy(y_pred,y_test):
    print("Accuracy Score:",accuracy_score(y_test, y_pred))

In [10]:
def submit(file,model):
    test_clean=pd.get_dummies(test.drop(columns=['id']))
    prediction=return_prediction(test_clean,model)
    submission=pd.DataFrame({
        'id':test['id'],
        'Irrigation_Need':prediction
    })
    submission.to_csv(file,index=False)

In [167]:
x_train,y_train,x_test,y_test=clean_dataset()
model=createDecisionTree(x_train,y_train)
y_prediction=return_prediction(x_test,model)
print_accuracy(y_prediction,y_test)
submit('decision_tree.csv',model)

Accuracy Score: 0.975


    K MEANS CLUSTERING

In [36]:
def classify_value(val):
    val*=3
    if val<=1:
        return 'Low'
    elif val>=3:
        return 'High'
    return 'Medium'

In [105]:
mapping = {'Low': 0, 'Medium': 1, 'High': 2}
revmapping = {0: 'Low', 1: 'Medium', 2: 'High'}
x_train, y_train, x_test, y_test = clean_dataset()
y_train = y_train.map(mapping)
#y_test = y_test.map(mapping)
model = KMeans(n_clusters=100, init='k-means++', n_init=10, max_iter=300, random_state=7)
model.fit(x_train)
cluster = model.predict(x_train)
kmeansd = pd.DataFrame({'Irrigation_Need': y_train.values, "Cluster": cluster})
kmeansdavg = np.floor((kmeansd.groupby('Cluster')['Irrigation_Need'].mean() * 2.99)).astype(int)
y_pred = model.predict(x_test)
y_pred = pd.Series(y_pred).map(kmeansdavg).map(revmapping)
#print_accuracy(y_pred,y_test)
y_pred = pd.DataFrame(
    {
        'id': train.iloc[10000:20000]['id'].values,
        'Irrigation_Need': y_pred.values
    }
)



In [112]:
y_pred = model.predict(pd.get_dummies(test.drop(columns=['id'])))
y_pred = pd.Series(y_pred).map(kmeansdavg).map(revmapping)
y_pred = pd.DataFrame(
    {
        'id': test['id'],
        'Irrigation_Need': y_pred.values
    }
)

y_pred.to_csv('kmeans.csv',index=False)


linear regression

In [160]:
from sklearn.linear_model import LinearRegression
x_train,y_train,x_test,y_test=clean_dataset()
x_test=pd.get_dummies(test).drop(columns=['id'])
model = LinearRegression()
model.fit(x_train, y_train.map(mapping)) 
y_prediction=return_prediction(x_test,model)
y_prediction=pd.Series(np.round(y_prediction*3)).map(revmapping)
#print_accuracy(y_prediction,y_test)
y_prediction = pd.DataFrame(
    {
        'id': test['id'],
        'Irrigation_Need': y_prediction.values
    }
)
y_prediction
#submit('hello.csv',model)
#x_test
y_prediction.to_csv("linear regression.csv",index=False)

Naive Baiyes

In [163]:
train.shape

(630000, 21)

In [166]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

x_train,y_train,x_test,y_test=clean_dataset()
model=GaussianNB()
model.fit(x_train,y_train)
y_prediction=return_prediction(x_test,model)
print_accuracy(y_prediction,y_test)
submit('guassian_naive_bayes.csv',model)


Accuracy Score: 0.7955
